# MSL Recognition — Column 2 Run (MSL Gloss Labels)

**Dataset:** MSL4Emergency (558 videos)  
**Label column:** Column 2 — MSL gloss sequence (e.g. `မီး ငြှိမ်း`)  
**Models:** BiLSTM + Attention | Transformer Encoder | ST-GCN  
**Config:** `config/config_kaggle.yaml` (batch_size=64, num_workers=4)  

Pipeline:
```
01_prepare_data  →  02_extract_keypoints  →  03_augment_data
→  04_train (bilstm)  →  04_train (transformer)  →  04_train (stgcn)
→  05_evaluate (all 3)  →  Summary table
```

In [1]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — only change this cell to switch experiments
# ═══════════════════════════════════════════════════════════════
LABEL_COL   = 2                          # 1=Myanmar text, 2=MSL gloss
CONFIG      = 'config/config_kaggle.yaml'
EXP_PREFIX  = f'exp_col{LABEL_COL}'     # exp_col2_bilstm, exp_col2_transformer, ...

VIDEOS_INPUT = '/kaggle/input/datasets/theinkyawlwin/msl4emergency-videos'
CODE_INPUT   = '/kaggle/input/datasets/theinkyawlwin/msl4emergency-code'

print(f'Label column : {LABEL_COL}')
print(f'Exp prefix   : {EXP_PREFIX}')
print(f'Config       : {CONFIG}')

Label column : 2
Exp prefix   : exp_col2
Config       : config/config_kaggle.yaml


## Step 0 — GPU & Environment Check

In [2]:
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i} : {props.name}  ({props.total_memory // 1024**2} MiB)')
!nvidia-smi

PyTorch  : 2.10.0+cu128
CUDA     : True
  GPU 0 : Tesla T4  (14911 MiB)
  GPU 1 : Tesla T4  (14911 MiB)
Sat Jul  4 09:14:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                   

## Step 1 — Install Dependencies

In [3]:
# Install dependencies (mediapipe 0.10.14 for legacy solutions + protobuf upgrade)
!pip install --upgrade protobuf
!pip install -q \
    mediapipe==0.10.14 \
    torchmetrics>=1.2.0 \
    seaborn>=0.13.0 \
    tqdm>=4.66.0
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.

## Step 2 — Setup Working Directory

In [4]:
import os, shutil
from pathlib import Path

# Copy pipeline code (src/, scripts/, config/, tools/) to working dir
print('Copying code from dataset...')
shutil.copytree(CODE_INPUT, '.', dirs_exist_ok=True)
print('  ✓ Code copied')

# Create data/ directory
os.makedirs('data', exist_ok=True)
os.makedirs('results/logs', exist_ok=True)

# Symlink videos — avoids copying 4.3 GB!
videos_src = f'{VIDEOS_INPUT}/video'
videos_dst = 'data/videos'
if not os.path.exists(videos_dst):
    os.symlink(videos_src, videos_dst)
    print(f'  ✓ Symlinked: {videos_dst} → {videos_src}')
else:
    print(f'  ✓ Symlink already exists: {videos_dst}')

# Copy annotations.txt
ann_src = f'{VIDEOS_INPUT}/annotations.txt'
ann_dst = 'data/annotations.txt'
shutil.copy(ann_src, ann_dst)
print(f'  ✓ Copied annotations.txt')

# Verify
n_videos = len(list(Path('data/videos').glob('*.mp4')))
print(f'\n  Videos found   : {n_videos}')
print(f'  Annotations    : {sum(1 for _ in open(ann_dst))}')

# Make scripts executable
!chmod +x scripts/*.sh
print('  ✓ Scripts ready')

Copying code from dataset...
  ✓ Code copied
  ✓ Symlinked: data/videos → /kaggle/input/datasets/theinkyawlwin/msl4emergency-videos/video
  ✓ Copied annotations.txt

  Videos found   : 558
  Annotations    : 558
  ✓ Scripts ready


## Step 3 — Prepare Data (label_col=1)

In [5]:
%%bash -s "$LABEL_COL" "$CONFIG"
LABEL_COL=$1
CONFIG=$2
LABEL_COL=$LABEL_COL CONFIG=$CONFIG bash scripts/01_prepare_data.sh

 Step 1 — Data Preparation
 Video dir       : data/videos
 Annotation file : data/annotations.txt
 Config          : config/config_kaggle.yaml
 Label column    : 2 (1=Myanmar text, 2=MSL gloss)

 Found 0 videos and 558 annotation lines

2026-07-04 09:14:48 | INFO     | prepare_data | ============================================================
2026-07-04 09:14:48 | INFO     | prepare_data | MSL Data Preparation
2026-07-04 09:14:48 | INFO     | prepare_data | Label column  : 2 → msl_gloss (MSL gloss)
2026-07-04 09:14:48 | INFO     | prepare_data | ============================================================
2026-07-04 09:14:48 | INFO     | prepare_data | Parsing annotation file: data/annotations.txt
2026-07-04 09:14:48 | INFO     | prepare_data | Total annotation records: 558
2026-07-04 09:14:48 | INFO     | prepare_data | Vocabulary built: 541 unique classes
2026-07-04 09:14:48 | INFO     | prepare_data | Label map saved → data/label_map.json
2026-07-04 09:14:48 | INFO     | prepare_da

## Step 4 — Extract MediaPipe Keypoints (~20 min)
Complexity=2 (most accurate). Already-extracted files are skipped on re-run.

In [6]:
!bash scripts/02_extract_keypoints.sh

 Step 2 — MediaPipe Holistic Keypoint Extraction
 Video dir     : data/videos
 Output dir    : data/keypoints
 Complexity    : 2
 Overwrite     : false

 Total videos  : 0

Extracting keypoints:   0%|          | 0/558 [00:00<?, ?video/s]INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1783156508.929858     131 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783156508.964520     132 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783156509.036405     125 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783156509.196592     125 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support f

## Step 5 — Augment Data (aug_factor=20, all-class design)

In [7]:
%%bash -s "$CONFIG"
CONFIG=$1
CONFIG=$CONFIG bash scripts/03_augment_data.sh

 Step 3 — All-Class Offline Data Augmentation
 Strategy    : all 558 videos → train/val/test by aug type
 Aug factor  : 20x
   Train     : 19 aug copies × 558 = 10602 samples
   Val       : 1 aug copy × 558 = 558 samples
   Test      : 558 originals (no augmentation)
 Class overlap: 100% across all splits

Augmenting all classes: 100%|██████████| 558/558 [00:31<00:00, 17.92sign/s]

  Augmentation Complete (all-class design)
  Original videos   : 558
  Aug factor        : 20  (train uses 19, val uses 1)
  Train samples     : 10602  (augmented, all classes)
  Val   samples     : 558   (augmented, all classes)
  Test  samples     : 558   (original,  all classes)
  Class overlap     : 100% across all splits
  Manifest          : data/augmented/augmented_manifest.json


 Augmentation done!  Manifest → data/augmented/augmented_manifest.json
 Next: bash scripts/04_train.sh


## Step 6 — Train BiLSTM + Attention

In [8]:
%%bash -s "$CONFIG" "$EXP_PREFIX"
CONFIG=$1
EXP_PREFIX=$2
CONFIG=$CONFIG bash scripts/04_train.sh bilstm ${EXP_PREFIX}_bilstm

 Training MSL Recognition Model
 Model      : bilstm
 Experiment : exp_col2_bilstm
 Config     : config/config_kaggle.yaml

 Data: train=10602, val=558, test=558 (all-class design)

[GPU]
Tesla T4, 15360 MiB, 14909 MiB, 39
Tesla T4, 15360 MiB, 14909 MiB, 40

2026-07-04 11:23:25 | INFO     | train | Experiment: exp_col2_bilstm  Model: bilstm
2026-07-04 11:23:25 | INFO     | train | Classes: 556
2026-07-04 11:23:25 | INFO     | train | Using all-class manifest: train=10602, val=558, test=558
2026-07-04 11:23:26 | INFO     | train | Model: bilstm  Params: 4,813,806
2026-07-04 11:23:31 | INFO     | train | Starting training…

[DataLoaders]
  Train: 10602 samples, 165 batches
  Val:   558   samples, 9 batches
  Test:  558  samples, 9 batches

2026-07-04 11:23:43 | INFO     | train | Epoch   1/150 | train loss=6.9903 top1=0.21% | val loss=6.5774 top1=0.52% top5=2.05% | lr=5.45e-05 | 12s
2026-07-04 11:23:44 | INFO     | train |   ★ New best val top-1: 0.52%
2026-07-04 11:23:55 | INFO     | tr

## Step 7 — Train Transformer Encoder

In [9]:
%%bash -s "$CONFIG" "$EXP_PREFIX"
CONFIG=$1
EXP_PREFIX=$2
CONFIG=$CONFIG bash scripts/04_train.sh transformer ${EXP_PREFIX}_transformer

 Training MSL Recognition Model
 Model      : transformer
 Experiment : exp_col2_transformer
 Config     : config/config_kaggle.yaml

 Data: train=10602, val=558, test=558 (all-class design)

[GPU]
Tesla T4, 15360 MiB, 14909 MiB, 74
Tesla T4, 15360 MiB, 14909 MiB, 34

2026-07-04 11:44:14 | INFO     | train | Experiment: exp_col2_transformer  Model: transformer
2026-07-04 11:44:14 | INFO     | train | Classes: 556
2026-07-04 11:44:15 | INFO     | train | Using all-class manifest: train=10602, val=558, test=558
2026-07-04 11:44:16 | INFO     | train | Model: transformer  Params: 3,362,030
2026-07-04 11:44:17 | INFO     | train | Starting training…

[DataLoaders]
  Train: 10602 samples, 165 batches
  Val:   558   samples, 9 batches
  Test:  558  samples, 9 batches

2026-07-04 11:44:27 | INFO     | train | Epoch   1/150 | train loss=6.3676 top1=0.22% | val loss=6.3542 top1=0.00% top5=1.22% | lr=5.45e-05 | 11s
2026-07-04 11:44:28 | INFO     | train |   ★ New best val top-1: 0.00%
2026-07-04

## Step 8 — Train ST-GCN

In [10]:
%%bash -s "$CONFIG" "$EXP_PREFIX"
CONFIG=$1
EXP_PREFIX=$2
CONFIG=$CONFIG bash scripts/04_train.sh stgcn ${EXP_PREFIX}_stgcn

 Training MSL Recognition Model
 Model      : stgcn
 Experiment : exp_col2_stgcn
 Config     : config/config_kaggle.yaml

 Data: train=10602, val=558, test=558 (all-class design)

[GPU]
Tesla T4, 15360 MiB, 14909 MiB, 75
Tesla T4, 15360 MiB, 14909 MiB, 34

2026-07-04 11:56:25 | INFO     | train | Experiment: exp_col2_stgcn  Model: stgcn
2026-07-04 11:56:26 | INFO     | train | Classes: 556
2026-07-04 11:56:26 | INFO     | train | Using all-class manifest: train=10602, val=558, test=558
2026-07-04 11:56:26 | INFO     | train | Model: stgcn  Params: 2,780,143
2026-07-04 11:56:27 | INFO     | train | Starting training…

[DataLoaders]
  Train: 10602 samples, 165 batches
  Val:   558   samples, 9 batches
  Test:  558  samples, 9 batches

2026-07-04 11:58:38 | INFO     | train | Epoch   1/150 | train loss=6.9783 top1=0.12% | val loss=6.5573 top1=0.17% top5=0.94% | lr=5.45e-05 | 131s
2026-07-04 11:58:39 | INFO     | train |   ★ New best val top-1: 0.17%
2026-07-04 12:00:48 | INFO     | train 

## Step 9 — Evaluate All 3 Models

In [11]:
%%bash -s "$CONFIG" "$EXP_PREFIX"
CONFIG=$1; EXP_PREFIX=$2
CONFIG=$CONFIG bash scripts/05_evaluate.sh bilstm      ${EXP_PREFIX}_bilstm
CONFIG=$CONFIG bash scripts/05_evaluate.sh transformer ${EXP_PREFIX}_transformer
CONFIG=$CONFIG bash scripts/05_evaluate.sh stgcn       ${EXP_PREFIX}_stgcn

 Evaluation — bilstm / exp_col2_bilstm
 Checkpoint : results/exp_col2_bilstm/checkpoints/best.pth
 Output dir : results/exp_col2_bilstm/evaluation

── Validation set ──────────────────────────────────────────
2026-07-04 15:52:17 | INFO     | evaluate | Loaded checkpoint: results/exp_col2_bilstm/checkpoints/best.pth  Model: bilstm
2026-07-04 15:52:17 | INFO     | evaluate | Evaluating 558 samples from 'val' split
2026-07-04 15:52:18 | INFO     | evaluate | Model loaded. Evaluating 558 samples…

───────────────────────────────────────────────────────
  Evaluation Results [VAL]
───────────────────────────────────────────────────────
  Samples       : 558
  Classes       : 556
  Top-1 Accuracy: 95.16%
  Top-5 Accuracy: 96.06%
  Precision (M) : 92.72%
  Recall    (M) : 95.14%
  F1        (M) : 93.53%
───────────────────────────────────────────────────────

Top-10 classes by F1:
  မီး ။                                F1=1.000  Prec=1.000  Rec=1.000  Support=1
  မီး ငြှိမ်း ။                 

## Step 10 — Results Summary

In [12]:
import json
from pathlib import Path

label_name = 'Myanmar Text (Col 1)' if LABEL_COL == 1 else 'MSL Gloss (Col 2)'

exps = [
    (f'{EXP_PREFIX}_bilstm',       'BiLSTM + Attention'),
    (f'{EXP_PREFIX}_transformer',  'Transformer Encoder'),
    (f'{EXP_PREFIX}_stgcn',        'ST-GCN'),
]

print(f'\n══════════════════════════════════════════════════════════')
print(f'  MSL Recognition Results — Label: {label_name}')
print(f'══════════════════════════════════════════════════════════')
print(f"{'Model':<24} {'Val Top-1':>10} {'Test Top-1':>11} {'Test F1':>9}")
print('─' * 58)

for exp_dir, model_name in exps:
    base = Path('results') / exp_dir
    val_m  = base / 'evaluation' / 'metrics_val.json'
    test_m = base / 'evaluation' / 'metrics_test.json'
    val_top1 = test_top1 = test_f1 = '—'
    if val_m.exists():
        d = json.load(open(val_m))
        val_top1  = f"{d['top1_accuracy']*100:.2f}%"
    if test_m.exists():
        d = json.load(open(test_m))
        test_top1 = f"{d['top1_accuracy']*100:.2f}%"
        test_f1   = f"{d['f1_macro']*100:.2f}%"
    print(f"  {model_name:<22} {val_top1:>10} {test_top1:>11} {test_f1:>9}")

print('─' * 58)
print(f'  Results saved in: /kaggle/working/results/')
print(f'══════════════════════════════════════════════════════════')


══════════════════════════════════════════════════════════
  MSL Recognition Results — Label: MSL Gloss (Col 2)
══════════════════════════════════════════════════════════
Model                     Val Top-1  Test Top-1   Test F1
──────────────────────────────────────────────────────────
  BiLSTM + Attention         95.16%     100.00%   100.00%
  Transformer Encoder        96.95%     100.00%   100.00%
  ST-GCN                     94.27%     100.00%   100.00%
──────────────────────────────────────────────────────────
  Results saved in: /kaggle/working/results/
══════════════════════════════════════════════════════════
